Jugando con los datos del case study based on Lambrechts (2008)

In [ ]:
!pip install dice-ml

  Using cached dice_ml-0.11-py3-none-any.whl.metadata (20 kB)
  Using cached jsonschema-4.23.0-py3-none-any.whl.metadata (7.9 kB)
  Using cached pandas-1.5.3-cp312-cp312-macosx_14_0_arm64.whl
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached raiutils-0.4.2-py3-none-any.whl.metadata (1.4 kB)
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached jsonschema_specifications-2024.10.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached referencing-0.36.2-py3-none-any.whl.metadata (2.8 kB)
  Using cached charset_normalizer-3.4.1-cp312-cp312-macosx_10_13_universal2.whl.metadata (35 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
  Using cached urllib3-2.3.0-py3-none-any.whl.metadata (6.5 kB)
Using cached dice_ml-0.11-py3-none-any.whl (2.5 MB)
Using cached raiutils-0.4.2-py3-none-any.whl (17 kB)
Using cached jsonschema-4.23.0-py3-none-any.whl (88 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [31]:
!pip install xgboost

  Using cached xgboost-2.1.4-py3-none-macosx_12_0_arm64.whl.metadata (2.1 kB)
Using cached xgboost-2.1.4-py3-none-macosx_12_0_arm64.whl (1.9 MB)


In [4]:
## Libraries

# Sklearn imports
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV


# DiCE imports
import dice_ml
from dice_ml.utils import helpers  # helper functions

# pandas and numpy
import pandas as pd
import numpy as np

In [6]:
# Carga del dataset de simulaciones
data=pd.read_csv('./data/simulation_EV0.75_5-rand.csv',index_col=0)
data['critical_path']=data['critical_path'].astype('str')

# Expected time of the project 13
data['delay']=(data['duration']>13)*1

for i in range(1, 9):
    col_name = f't{i}_on_cp'
    data[col_name] = data['critical_path'].apply(lambda x: 1 if str(i) in x else 0)

In [65]:
data.head(2)

,duration1,duration2,duration3,duration4,duration5,duration6,duration7,duration8,duration@1,duration@2,...,critical_path,delay,t1_on_cp,t2_on_cp,t3_on_cp,t4_on_cp,t5_on_cp,t6_on_cp,t7_on_cp,t8_on_cp
0,1.890262,4.981053,6.176876,2.525289,4.874607,4.314132,4.488550,2.264787,1.890262,4.981053,...,368,0,0,0,1,0,0,1,0,1
1,1.923657,2.671438,5.377826,2.293507,17.989498,4.509809,8.463748,2.082815,1.923657,2.671438,...,259,1,0,1,0,0,1,0,0,0


# Gradient boosting clasificator

In [15]:
# random seed
seed=112358

### RETRASO en función de estado de las tareas@ (y=D^BAC)
DBACdelay~ {activity i's duration at 75%EV} i=1,...,8

In [16]:
y=data.loc[:,'delay']
X=data.loc[:,['duration@1','duration@2', 'duration@3','duration@4', 'duration@5','duration@6', 'duration@7','duration@8']]

subsetdata=data.loc[:,['duration@1','duration@2', 'duration@3','duration@4', 'duration@5','duration@6', 'duration@7','duration@8','delay']]

In [17]:
# outer kfolds
kfold =StratifiedKFold(n_splits=5, random_state=seed,shuffle=True)

In [18]:
# results
results = pd.DataFrame([],columns=['model','kf','Accuracy'])

Para hacer model selection y tuneado de hiper parámetros haríamos como en sheva_model_seletion.

Voy a partir del gradient boosting y los parámetros que ya ajustamos:
 {'max_depth': 7, 'n_estimators': 100}

In [19]:
# Training 
mdC_dbac = GradientBoostingClassifier(max_depth=7, n_estimators=100,random_state=seed)
mdC_dbac.fit(X,y)

GradientBoostingClassifier(max_depth=7, random_state=112358)

In [ ]:
# Training 
mdC_dbac = GradientBoostingClassifier(max_depth=7, n_estimators=100,random_state=seed)
mdC_dbac.fit(X,y)

GradientBoostingClassifier(max_depth=7, random_state=112358)

In [20]:
accuracy_score(y,mdC_dbac.predict(X))

0.89156

## Abajo tenemos la primera prueba que hice con generación RANDOM de la categoría model agnostic ##

DiCE can generate counterfactual examples using the following methods.

1) Model-agnostic methods

- Randomized sampling

- KD-Tree (for counterfactuals within the training data)

- Genetic algorithm


2) Gradient-based methods

- An explicit loss-based method described in Mothilal et al. (2020) (Default for deep learning models).

- A Variational AutoEncoder (VAE)-based method described in Mahajan et al. (2019) (see the BaseVAE notebook).

### 1.1 MODEL-AGNOSTIC, RANDOM

In [21]:
 #construct a data object for DiCE. 
d = dice_ml.Data(dataframe=subsetdata, continuous_features=['duration@1','duration@2', 'duration@3','duration@4', 'duration@5','duration@6', 'duration@7','duration@8'], outcome_name='delay')

# initiate DiCE
#We now initialize the DiCE explainer, which needs a dataset and a model. DiCE provides local explanation for the model m and requires an query input whose outcome needs to be explained.
# Using sklearn backend
m = dice_ml.Model(model=mdC_dbac, backend="sklearn")
# Using method=random for generating CFs
exp = dice_ml.Dice(d, m, method="random")

Vamos a generarle CF a la segunda instancia que va con retraso

In [22]:
subsetdata[1:2]

,duration@1,duration@2,duration@3,duration@4,duration@5,duration@6,duration@7,duration@8,delay
1,1.923657,2.671438,5.377826,2.293507,5.580528,2.874139,4.034801,0.0,1


In [23]:
type(subsetdata[1:2])

pandas.core.frame.DataFrame

In [24]:
dice_exp = exp.generate_counterfactuals(X[1:2], total_CFs=8, desired_class="opposite")
dice_exp.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00,  4.95it/s]

Query instance (original outcome : 1)


,duration@1,duration@2,duration@3,duration@4,duration@5,duration@6,duration@7,duration@8,delay
0,1.923657,2.671438,5.377826,2.293507,5.580528,2.874139,4.034801,0.0,1



Diverse Counterfactual set (new outcome: 0)


,duration@1,duration@2,duration@3,duration@4,duration@5,duration@6,duration@7,duration@8,delay
0,-,-,-,4.6792476,-,0.3,-,-,0.0
1,-,-,2.9948292,-,-,1.8,-,-,0.0
2,-,-,-,-,-,-,2.4605198,-,0.0
3,-,-,-,-,-,-,2.7508809,-,0.0
4,-,-,-,-,3.4991563,0.1,-,-,0.0
5,-,-,-,-,-,0.8,-,-,0.0
6,0.7538371,-,4.5743211,-,-,-,-,-,0.0
7,-,-,-,-,-,0.0,-,-,0.0


In [25]:
# get MAD
mads = d.get_mads(normalized=True)

# create feature weights
feature_weights = {}
for feature in mads:
    feature_weights[feature] = round(1/mads[feature], 2)
print(feature_weights)

{'duration@1': 13.73, 'duration@2': 12.88, 'duration@3': 11.37, 'duration@4': 12.97, 'duration@5': 13.63, 'duration@6': 10.19, 'duration@7': 11.75, 'duration@8': inf}


/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_43879/1464587245.py:7: RuntimeWarning: divide by zero encountered in divide
  feature_weights[feature] = round(1/mads[feature], 2)


He añadido dummy columns que nos digan si las tareas pertenecen al CP

### 2.1 GRADIENT METHODS. Explicit loss-based method described in Mothilal et al. (2020). Only for differenciable clasiffiers

No funciona con GradientBoostingClassifier: gradient is only supported for differentiable neural network models
Tampoco con XGBClassifier

In [35]:
# https://github.com/interpretml/DiCE/blob/main/docs/source/notebooks/DiCE_getting_started.ipynb
#  
#construct a data object for DiCE. 
d = dice_ml.Data(dataframe=subsetdata, continuous_features=['duration@1','duration@2', 'duration@3','duration@4', 'duration@5','duration@6', 'duration@7','duration@8'], outcome_name='delay')

# initiate DiCE
#We now initialize the DiCE explainer, which needs a dataset and a model. DiCE provides local explanation for the model m and requires an query input whose outcome needs to be explained.
# Using sklearn backend
m = dice_ml.Model(model=mdC_dbac, backend="sklearn")
# Using method=random for generating CFs
exp = dice_ml.Dice(d, m, method="gradient")

UserConfigValidationException: gradient is only supported for differentiable neural network models. Please choose one of random, genetic or kdtree

In [ ]:
#generation
dice_exp = exp.generate_counterfactuals(X[1:2], total_CFs=8, desired_class="opposite")
dice_exp.visualize_as_dataframe(show_only_changes=True)

In [ ]:
# Training XGBClassifier
from xgboost.sklearn import XGBClassifier
mdC_dbacXGBClassifier = XGBClassifier(max_depth=7, n_estimators=100,random_state=seed)
mdC_dbacXGBClassifier.fit(X,y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, random_state=112358, ...)

In [ ]:
m2 = dice_ml.Model(model=mdC_dbacXGBClassifier, backend="sklearn")
# Using method=random for generating CFs
exp = dice_ml.Dice(d, m2, method="gradient")

UserConfigValidationException: gradient is only supported for differentiable neural network models. Please choose one of random, genetic or kdtree